In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [2]:
seed = 42
prefix_path = "../"

In [3]:
data = pd.read_csv(prefix_path + "data/raw/Student_Performance.csv", index_col=0)

In [4]:
data.head()

,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
Hours Studied,,,,,
7,99,Yes,9,1,91.0
4,82,No,4,2,65.0
8,51,Yes,7,2,45.0
5,52,Yes,5,2,36.0
7,75,No,8,5,66.0


In [7]:
X, y = np.array(data.drop(columns=["Performance Index"])), np.array(data["Performance Index"])

In [9]:
X.shape, y.shape

((10000, 4), (10000,))

In [12]:
X, y

(array([[99, 'Yes', 9, 1],
        [82, 'No', 4, 2],
        [51, 'Yes', 7, 2],
        ...,
        [83, 'Yes', 8, 5],
        [97, 'Yes', 7, 0],
        [74, 'No', 8, 1]], shape=(10000, 4), dtype=object),
 array([91., 65., 45., ..., 74., 95., 64.], shape=(10000,)))

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=seed)

In [14]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 4), (2000, 4), (8000,), (2000,))

In [15]:
MAX_ITERATIONS = 2e5
TOLERANCE = 1e-2

In [18]:
X_train

array([[49, 'No', 7, 5],
       [48, 'Yes', 7, 6],
       [81, 'No', 7, 2],
       ...,
       [48, 'No', 7, 6],
       [47, 'No', 9, 0],
       [46, 'No', 6, 6]], shape=(8000, 4), dtype=object)

In [19]:
import numpy as np
import pandas as pd

def normalization(data) -> np.array:
    """
    Normalize numeric columns of a dataset to range [0, 1],
    ignoring categorical (string/object) columns.

    Parameters
    ----------
    data : np.array or pd.DataFrame
        2D dataset with samples and features.
    
    Returns
    -------
    np.array
        Array with normalized numeric columns and original categorical columns preserved.
    """
    if isinstance(data, np.ndarray):
        data = pd.DataFrame(data)
    
    # Identify the numerical and categorical columns
    num_cols = data.select_dtypes(include=[np.number]).columns
    cat_cols = data.select_dtypes(exclude=[np.number]).columns

    # Create as copy of dataframe
    data_norm = data.copy()

    # Normalize only the numeric columns
    for col in num_cols:
        min_val = data[col].min()
        max_val = data[col].max()
        diff = max_val - min_val
        if diff != 0:
            data_norm[col] = (data[col] - min_val) / diff

    # Returns the same type expected, a numpy array
    return data_norm.to_numpy() if isinstance(data, pd.DataFrame) else data_norm


In [20]:
X_train_norm = normalization(X_train)
y_train_norm = normalization(y_train)

In [22]:
X_train

array([[49, 'No', 7, 5],
       [48, 'Yes', 7, 6],
       [81, 'No', 7, 2],
       ...,
       [48, 'No', 7, 6],
       [47, 'No', 9, 0],
       [46, 'No', 6, 6]], shape=(8000, 4), dtype=object)

In [21]:
X_train_norm

array([[49, 'No', 7, 5],
       [48, 'Yes', 7, 6],
       [81, 'No', 7, 2],
       ...,
       [48, 'No', 7, 6],
       [47, 'No', 9, 0],
       [46, 'No', 6, 6]], shape=(8000, 4), dtype=object)

In [ ]:

def standardization(data: np.array) -> np.array:
    mean_data = np.mean(data)
    std_data = np.std(data)
    return (data - mean_data) / std_data if std_data != 0 else data

In [ ]:

X_train_z = standardization(X_train)
y_train_z = standardization(y_train)

In [ ]:
def f(X: np.array, w: float, b: float) -> float:
    return w * X + b

In [ ]:
def loss_function(X: np.array, y: np.array, w: float, b: float, metric: str) -> float:
    # Mean Squared Error (MSE)
    if metric.strip().lower() == "mse":
        return (1/(2 * X.shape[0])) * np.sum( (f(X, w, b) - y)**2 )

In [ ]:
def update_coefficients(X: np.array, y: np.array, w: float, b: float, lr: float, metric: str, method: str) -> float:
    # Gradient Descent (GD)
    if method.strip().lower() == "gd":
        def d_loss_function(X: np.array, y: np.array, w: float, b: float, metric: str) -> float:
            # Derivatives of Mean Squared Error
            if metric.strip().lower() == "mse":
                dw = (1/X.shape[0]) * np.sum( (f(X, w, b) - y) * X )
                db = (1/X.shape[0]) * np.sum( f(X, w, b) - y )
                return dw, db
    
        dw, db = d_loss_function(X, y, w, b, metric)
        aux_w = w - lr * dw
        aux_b = b - lr * db
        return aux_w, aux_b

In [ ]:
def gradient_descent(X: np.array, y: np.array, lr: float, metric: str):
    # Defining returnning structure
    hist_cost = list()
    hist_w_b = list()

    i = 1 # Iterations counter
    # Generating initial values for w and b
    w, b = 6, 7
    # Initial cost
    cost = loss_function(X, y, w, b, metric)
    hist_cost.append(cost)
    while (cost > TOLERANCE) and i < MAX_ITERATIONS:
        w, b = update_coefficients(X, y, w, b, lr, metric, method="gd")    # Update the coefficients of f
        cost = loss_function(X, y, w, b, metric)             # The total cost related to new coefficients
        # Saving the history
        hist_w_b.append((w, b))
        hist_cost.append(cost)
        i += 1
    
    return hist_w_b, hist_cost, i

In [ ]:
def linear_regression(X: np.array, y: np.array, lr: float, metric: str, method: str) -> float:
    if method.strip().lower() == "gd":
        return gradient_descent(X, y, lr, metric)

In [ ]:
X_train_norm = normalization(X_train)
y_train_norm = normalization(y_train)

X_train_z = standardization(X_train)
y_train_z = standardization(y_train)

In [ ]:
inputs = {"X": X_train_norm, "y": y_train_norm, "lr": 0.1, "metric": "mse", "method": "gd"}
list_w_b, list_cost, total_iterations = linear_regression(**inputs)

In [ ]:
len(list_cost), total_iterations

In [ ]:
def learning_curve(cost: list) -> None:
    x_len = len(cost)
    plt.plot(np.arange(x_len), cost, label="Loss")
    plt.xticks(ticks=np.arange(0, x_len+1, (MAX_ITERATIONS / 1e1)), rotation=45)
    plt.legend()

In [ ]:
learning_curve(list_cost)

In [ ]:
min(list_cost)

In [ ]:
X_test_norm = normalization(X_test)
y_test_norm = normalization(y_test)

X_test_z = standardization(X_test)
y_test_z = standardization(y_test)

In [ ]:
inputs = {"X": X_test_norm, "y": y_test_norm, "lr": 0.1, "metric": "mse", "method": "gd"}
list_w_b, list_cost, total_iterations = linear_regression(**inputs)

In [ ]:
learning_curve(list_cost)

In [ ]:
list_cost[-1]